In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install duckdb==1.4.4 polars==1.40.0 --quiet

In [ ]:
# ── Cell 2: Mount Drive and copy DB to local (Colab only) ─────────────────
from pathlib import Path
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    DRIVE_DB = Path("/content/drive/MyDrive/b3_data/db/b3_data.duckdb")
    LOCAL_DB = Path("/content/b3_data.duckdb")

    if not DRIVE_DB.exists():
        raise FileNotFoundError(
            f"Database not found at {DRIVE_DB}.\n"
            "Run colab_ingest.ipynb first to build the database."
        )

    print(f"Copying DB from Drive → {LOCAL_DB} ...")
    shutil.copy2(DRIVE_DB, LOCAL_DB)
    size_mb = LOCAL_DB.stat().st_size / 1_048_576
    print(f"Ready. DB size: {size_mb:.1f} MB")
    DB_PATH = LOCAL_DB
else:
    DB_PATH = Path("/data/b3.duckdb")
    print(f"Using local DB: {DB_PATH}")

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import duckdb
import pandas as pd
import plotly.graph_objects as go
from datetime import date, timedelta

In [ ]:
# ── Cell 4: Parameters — edit this cell to run a new analysis ─────────────
# ─────────────────────────────────────────────────────────────────────────
TICKER            = "PETR4"  # underlying stock ticker
START_DATE        = "2024-01-01"  # backtest start (first entry on/after this date)
END_DATE          = "2024-12-31"  # backtest end (no new entries after this date)
STRIKE_OFFSET_PCT = 0   # 0 = ATM; positive = OTM (e.g. 5 → CALL +5%, PUT -5%)
EXPIRY_OFFSET     = 1   # how many monthly expiries ahead the options expire
                        #   1 = next expiry (Jan→Feb), 3 = three months out (Jan→Apr)
ENTRY_STEP        = 1   # expiries between entries
                        #   1 = enter every month, 3 = enter every 3rd expiry
# ─────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Cell 5: Open connection ───────────────────────────────────────────────
con = duckdb.connect(str(DB_PATH))
con.execute("SET threads TO 2")
con.execute("SET memory_limit = '4GB'")
print("Connection open. DuckDB version:", duckdb.__version__)

In [ ]:
# ── Cell 6: Backtest loop ─────────────────────────────────────────────────
# Strategy: on each entry expiry date, buy ATM CALL + ATM PUT expiring
# EXPIRY_OFFSET monthly expiries ahead. Entries are spaced ENTRY_STEP
# expiries apart. Worst leg is a full loss; best leg is sold at its peak
# closing price.
# Strategy return = (winner_max / total_straddle_cost - 1) × 100
# Special case: if winner never rose above entry price → -100% (full loss).

warnings_log = []
results      = []

# ── 6a: fetch monthly expiry dates ───────────────────────────────────────
# Look-ahead window = EXPIRY_OFFSET months past END_DATE so target expiries
# are available even for the last entries in range.
expiry_rows = con.execute(f"""
    SELECT DISTINCT datven
    FROM cotacoes
    WHERE codneg LIKE '{TICKER[:4]}%'
      AND tpmerc IN ('070', '080')
      AND datven BETWEEN '{START_DATE}'::DATE
                     AND ('{END_DATE}'::DATE + INTERVAL '{EXPIRY_OFFSET * 35} days')
      AND NOT regexp_matches(codneg, 'W[1-5]$')
    ORDER BY datven
""").fetchall()

expiry_dates = [r[0] for r in expiry_rows]

if len(expiry_dates) < 2:
    raise ValueError(
        f"Need at least 2 monthly expiry dates starting from {START_DATE} "
        f"for {TICKER}. Check ticker and date range."
    )

print(f"Found {len(expiry_dates)} monthly expiry dates (including look-ahead).")

# ── 6b: build (entry_expiry, target_expiry) pairs ─────────────────────────
# Entry  = expiry[i]                  — trader buys on this day
# Target = expiry[i + EXPIRY_OFFSET]  — purchased options expire here
#          falls back to expiry[-1] when the offset exceeds available data
# Entries advance by ENTRY_STEP each iteration.
start_date_obj = date.fromisoformat(START_DATE)
end_date_obj   = date.fromisoformat(END_DATE)

periods = []
for i in range(0, len(expiry_dates), ENTRY_STEP):
    entry_expiry = expiry_dates[i]
    if not (start_date_obj <= entry_expiry <= end_date_obj):
        continue

    target_idx    = i + EXPIRY_OFFSET
    target_expiry = (
        expiry_dates[target_idx]
        if target_idx < len(expiry_dates)
        else expiry_dates[-1]
    )

    if target_expiry == entry_expiry:
        warnings_log.append(
            f"[{entry_expiry.strftime('%Y-%m')}] Target expiry equals entry date "
            f"({entry_expiry}) — no data beyond this point. Skipped."
        )
        continue

    periods.append((entry_expiry, target_expiry))

if not periods:
    raise ValueError(
        f"No valid periods found within [{START_DATE}, {END_DATE}] for {TICKER}."
    )

print(f"Periods to backtest: {len(periods)}")

# ── 6c: per-period loop ───────────────────────────────────────────────────
for entry_expiry, target_expiry in periods:

    period_label = entry_expiry.strftime("%Y-%m")

    # Step 1: get stock closing price and especi on entry date (= expiry day)
    row = con.execute(f"""
        SELECT preult, TRIM(especi)
        FROM cotacoes
        WHERE codneg = '{TICKER}'
          AND datpre = '{entry_expiry}'::DATE
          AND codbdi = '02'
    """).fetchone()

    if not row:
        warnings_log.append(
            f"[{period_label}] No stock quote for {TICKER} on {entry_expiry}. Skipped."
        )
        continue

    stock_price, stock_especi = row
    entry_date = entry_expiry

    # Step 2: select nearest-ATM CALL and PUT expiring on target_expiry
    row = con.execute(f"""
        WITH options_on_entry AS (
            SELECT
                codneg,
                tpmerc,
                preexe,
                ABS(preexe - CASE
                    WHEN tpmerc = '070' THEN {stock_price} * (1 + {STRIKE_OFFSET_PCT}/100.0)
                    WHEN tpmerc = '080' THEN {stock_price} * (1 - {STRIKE_OFFSET_PCT}/100.0)
                END) AS strike_diff
            FROM cotacoes
            WHERE datpre = '{entry_date}'::DATE
              AND tpmerc IN ('070', '080')
              AND codneg LIKE '{TICKER[:4]}%'
              AND SUBSTR(especi, 1, 2) = '{stock_especi[:2]}'
              AND NOT regexp_matches(codneg, 'W[1-5]$')
              AND datven = '{target_expiry}'::DATE
        ),
        best_call AS (
            SELECT codneg, preexe FROM options_on_entry
            WHERE tpmerc = '070' ORDER BY strike_diff LIMIT 1
        ),
        best_put AS (
            SELECT codneg, preexe FROM options_on_entry
            WHERE tpmerc = '080' ORDER BY strike_diff LIMIT 1
        )
        SELECT
            (SELECT codneg FROM best_call) AS call_ticker,
            (SELECT preexe FROM best_call) AS call_strike,
            (SELECT codneg FROM best_put)  AS put_ticker,
            (SELECT preexe FROM best_put)  AS put_strike
    """).fetchone()

    if not row or row[0] is None or row[2] is None:
        warnings_log.append(
            f"[{period_label}] No CALL+PUT found for {TICKER} "
            f"expiring {target_expiry} on entry {entry_date}. Skipped."
        )
        continue

    call_ticker, call_strike, put_ticker, put_strike = row

    # Step 3: entry closing prices for each leg
    call_entry_row = con.execute(f"""
        SELECT preult FROM cotacoes
        WHERE codneg = '{call_ticker}' AND tpmerc = '070'
          AND datpre = '{entry_date}'::DATE
    """).fetchone()

    put_entry_row = con.execute(f"""
        SELECT preult FROM cotacoes
        WHERE codneg = '{put_ticker}' AND tpmerc = '080'
          AND datpre = '{entry_date}'::DATE
    """).fetchone()

    if (not call_entry_row or not call_entry_row[0]
            or not put_entry_row or not put_entry_row[0]):
        warnings_log.append(
            f"[{period_label}] Zero or missing entry price for "
            f"{call_ticker}/{put_ticker} on {entry_date}. Skipped."
        )
        continue

    call_entry_price = call_entry_row[0]
    put_entry_price  = put_entry_row[0]

    # Step 4: max closing price over the holding period (entry → target expiry)
    call_max = con.execute(f"""
        SELECT MAX(preult) FROM cotacoes
        WHERE codneg = '{call_ticker}' AND tpmerc = '070'
          AND datpre BETWEEN '{entry_date}'::DATE AND '{target_expiry}'::DATE
    """).fetchone()[0] or call_entry_price

    put_max = con.execute(f"""
        SELECT MAX(preult) FROM cotacoes
        WHERE codneg = '{put_ticker}' AND tpmerc = '080'
          AND datpre BETWEEN '{entry_date}'::DATE AND '{target_expiry}'::DATE
    """).fetchone()[0] or put_entry_price

    # Step 5: determine winner and compute strategy return
    call_ratio = call_max / call_entry_price
    put_ratio  = put_max  / put_entry_price

    if call_ratio >= put_ratio:
        winner       = "CALL"
        winner_max   = call_max
        winner_entry = call_entry_price
    else:
        winner       = "PUT"
        winner_max   = put_max
        winner_entry = put_entry_price

    winner_return_pct = (winner_max / winner_entry - 1) * 100

    total_straddle_cost = call_entry_price + put_entry_price

    if winner_max <= winner_entry:
        # Neither option gained — full loss of premium paid
        strategy_return_pct = -100.0
    else:
        strategy_return_pct = (winner_max / total_straddle_cost - 1) * 100

    results.append({
        "period":               period_label,
        "stock_ticker":         TICKER,
        "entry_date":           entry_date,
        "expiry_date":          target_expiry,
        "call_ticker":          call_ticker,
        "put_ticker":           put_ticker,
        "call_strike":          call_strike,
        "put_strike":           put_strike,
        "call_entry_price":     call_entry_price,
        "put_entry_price":      put_entry_price,
        "call_max_price":       call_max,
        "put_max_price":        put_max,
        "winner":               winner,
        "winner_return_pct":    round(winner_return_pct, 2),
        "strategy_return_pct":  round(strategy_return_pct, 2),
    })

for w in warnings_log:
    print(f"WARNING: {w}")

print(f"\nCompleted: {len(results)} periods processed, {len(warnings_log)} skipped.")

In [ ]:
# ── Cell 7: Results table ─────────────────────────────────────────────────
if not results:
    print("No results to display. Check warnings above.")
else:
    df_results = pd.DataFrame(results)

    styled = (
        df_results.style
        .format({
            "call_strike":          "R$ {:.2f}",
            "put_strike":           "R$ {:.2f}",
            "call_entry_price":     "R$ {:.2f}",
            "put_entry_price":      "R$ {:.2f}",
            "call_max_price":       "R$ {:.2f}",
            "put_max_price":        "R$ {:.2f}",
            "winner_return_pct":    "{:.2f}%",
            "strategy_return_pct":  "{:.2f}%",
        })
        .bar(
            subset=["strategy_return_pct"],
            align="mid",
            color=["#ff4466", "#00cc44"],
        )
    )
    display(styled)

    total_periods = len(df_results)
    wins          = (df_results["strategy_return_pct"] > 0).sum()
    win_rate      = wins / total_periods * 100
    avg_return    = df_results["strategy_return_pct"].mean()
    best          = df_results.loc[df_results["strategy_return_pct"].idxmax()]
    worst         = df_results.loc[df_results["strategy_return_pct"].idxmin()]

    print(f"\n{'='*55}")
    print(f"  Backtest Summary: {TICKER} | {START_DATE} → {END_DATE}")
    print(f"{'='*55}")
    print(f"  Total periods  : {total_periods}")
    print(f"  Win rate       : {wins}/{total_periods} ({win_rate:.1f}%)")
    print(f"  Average return : {avg_return:+.2f}%")
    print(f"  Best trade     : {best['period']}  {best['strategy_return_pct']:+.2f}%  (winner: {best['winner']})")
    print(f"  Worst trade    : {worst['period']}  {worst['strategy_return_pct']:+.2f}%  (winner: {worst['winner']})")
    print(f"{'='*55}")

In [ ]:
# ── Cell 8: Bar chart — strategy return per period ────────────────────────
if 'df_results' not in dir() or df_results.empty:
    print("No results to chart.")
else:
    bar_colors = [
        "#00cc44" if v >= 0 else "#ff4466"
        for v in df_results["strategy_return_pct"]
    ]

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=df_results["expiry_date"].astype(str),
        y=df_results["strategy_return_pct"],
        marker_color=bar_colors,
        text=[f"{v:+.1f}%" for v in df_results["strategy_return_pct"]],
        textposition="outside",
        textfont=dict(color="white", size=11),
        customdata=df_results[["call_ticker", "put_ticker", "winner", "winner_return_pct"]].values,
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Strategy return: %{y:.2f}%<br>"
            "CALL: %{customdata[0]}<br>"
            "PUT:  %{customdata[1]}<br>"
            "Winner: %{customdata[2]} (%{customdata[3]:.1f}%)"
            "<extra></extra>"
        ),
    ))

    fig.add_hline(
        y=0,
        line=dict(color="rgba(255,255,255,0.3)", width=1, dash="dot"),
    )

    offset_label = f" | Offset: {STRIKE_OFFSET_PCT:+.0f}%" if STRIKE_OFFSET_PCT != 0 else ""

    fig.update_layout(
        title=dict(
            text=(
                f"{TICKER} — Straddle Backtest Strategy Return<br>"
                f"<sup>{START_DATE} → {END_DATE}"
                f"{offset_label}"
                f" | Expiry offset: {EXPIRY_OFFSET} | Entry step: {ENTRY_STEP}"
                f" | Win rate: {win_rate:.1f}%"
                f" | Avg: {avg_return:+.2f}%</sup>"
            ),
            font_color="white",
        ),
        plot_bgcolor="#1a1a2e",
        paper_bgcolor="#0d0d1a",
        font=dict(color="white"),
        xaxis=dict(
            title="Target Expiry Date",
            showgrid=False,
            tickfont=dict(color="white"),
        ),
        yaxis=dict(
            title="Strategy Return (%)",
            showgrid=True,
            gridcolor="rgba(255,255,255,0.08)",
            zeroline=False,
            ticksuffix="%",
        ),
        bargap=0.3,
        showlegend=False,
        hovermode="x",
        margin=dict(t=100, b=60, l=60, r=40),
    )

    display(fig)

In [ ]:
# ── Cell 9: Close connection and sync DB back to Drive ────────────────────
con.close()

if IN_COLAB:
    shutil.copy2(LOCAL_DB, DRIVE_DB)
    print(f"DB synced to Drive. Size: {DRIVE_DB.stat().st_size / 1_048_576:.1f} MB")
else:
    print("Connection closed.")